# AI-Publisher Docker Image Builder v2

Bu notebook, AI-Publisher platformu icin Docker imajlarini Google Colab'da Kaniko ile build eder.
Imajlar Google Drive'a `.tar.gz` olarak kaydedilir, yerel makinede `docker compose up` ile yuklenir.

### Degisiklikler (v2):
- **Video codec fix**: cogvideox, wan, ltx, hunyuan, svd, animatediff, wan25 -> libx264 + yuv420p
- **Colab runtime kalkti**: Bu notebook sadece build icin, runtime sunucu Docker container'larda
- **Force-Rebuild**: Video modelleri zorunlu rebuild edilir (codec fix icin)
- **Incremental**: Ses/efekt modelleri Drive'da varsa atlanir

---
Hucreleri sirayla calistirin. Her hucre bir oncekinin ciktisina baglidir.

In [ ]:
#@title 1. Runtime Kontrolu { display-mode: "form" }
import subprocess, os

print("Sistem durumu:")
try:
    r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True, timeout=5)
    if r.returncode == 0:
        print(f"  GPU: {r.stdout.strip()} (build icin gerekmez, CPU yeterli)")
    else:
        print("  GPU: yok (CPU build)")
except:
    print("  GPU: yok (CPU build)")

disk = subprocess.run(["df", "-h", "/content"], capture_output=True, text=True)
if disk.returncode == 0:
    parts = disk.stdout.strip().split("\n")[1].split()
    print(f"  Disk: {parts[2]} / {parts[1]}")

with open("/proc/meminfo") as f:
    for line in f:
        if line.startswith("MemTotal"):
            print(f"  RAM: {int(line.split()[1]) // 1024} MB")
            break
print("\nDevam etmek icin sonraki hucreye gec.")

In [ ]:
#@title 2. Drive Mount ve Bagimliliklar { display-mode: "form" }
import subprocess, sys, os

print("1/4 Drive baglaniyor...")
try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    print("  Google Drive bagli.")
except Exception as e:
    print(f"  Drive mount: {e}")

print("\n2/4 Kaniko binary indiriliyor...")
os.makedirs("/kaniko", exist_ok=True)
if os.path.exists("/kaniko/executor") and os.access("/kaniko/executor", os.X_OK):
    print("  Kaniko zaten mevcut ve calisir durumda.")
else:
    # Eski/kirik binary varsa sil
    if os.path.exists("/kaniko/executor"):
        os.remove("/kaniko/executor")
    urls = [
        "https://storage.googleapis.com/kaniko-releases/v1.23.2/kaniko-linux-amd64",
        "https://storage.googleapis.com/kaniko-releases/v1.21.1/kaniko-linux-amd64",
        "https://storage.googleapis.com/kaniko-releases/v1.20.0/kaniko-linux-amd64",
        "https://github.com/GoogleContainerTools/kaniko/releases/download/v1.23.2/kaniko-linux-amd64",
    ]
    ok = False
    for url in urls:
        ver = url.split("/")[-2]
        print(f"  Deneniyor: {ver}...")
        r = subprocess.run(["curl", "-L", "--connect-timeout", "15", "--max-time", "120",
                           "-o", "/kaniko/executor", url], capture_output=True)
        sz = os.path.getsize("/kaniko/executor") if os.path.exists("/kaniko/executor") else 0
        if r.returncode == 0 and sz > 1000000:
            subprocess.run(["chmod", "+x", "/kaniko/executor"])
            # Verify executable
            if os.access("/kaniko/executor", os.X_OK):
                subprocess.run(["ln", "-sf", "/kaniko/executor", "/usr/local/bin/kaniko"])
                print(f"  Kaniko kuruldu ({sz // 1024 // 1024} MB)")
                ok = True
                break
        if os.path.exists("/kaniko/executor"):
            os.remove("/kaniko/executor")
    if not ok:
        print("  Kaniko indirilemedi! build_all.sh basarisiz olur.")

print("\n3/4 Registry + pigz kontrol...")
if not os.path.exists("/usr/local/bin/registry"):
    r = subprocess.run(["curl", "-L", "--connect-timeout", "10", "--max-time", "60",
                       "-o", "/tmp/registry.tar.gz",
                       "https://github.com/distribution/distribution/releases/download/v2.8.3/registry_2.8.3_linux_amd64.tar.gz"],
                      capture_output=True)
    if r.returncode == 0 and os.path.getsize("/tmp/registry.tar.gz") > 100000:
        subprocess.run(["tar", "-xzf", "/tmp/registry.tar.gz", "-C", "/tmp", "registry"])
        if os.path.exists("/tmp/registry"):
            subprocess.run(["cp", "/tmp/registry", "/usr/local/bin/registry"])
        os.remove("/tmp/registry.tar.gz")
        print("  Registry kuruldu.")
    else:
        print("  Registry indirilemedi.")
else:
    print("  Registry mevcut.")

r = subprocess.run(["which", "pigz"], capture_output=True)
if r.returncode != 0:
    r = subprocess.run(["apt-get", "install", "-y", "-qq", "pigz"], capture_output=True)
    print("  pigz kuruldu." if r.returncode == 0 else "  pigz yok, gzip kullanilacak.")
else:
    print("  pigz mevcut.")

print("\n4/4 Registry config...")
os.makedirs("/etc/docker/registry", exist_ok=True)
with open("/etc/docker/registry/config.yml", "w") as f:
    f.write("version: 0.1\nlog:\n  level: error\nstorage:\n  filesystem:\n    rootdirectory: /var/lib/registry\nhttp:\n  addr: :5000\n")
os.makedirs("/var/lib/registry", exist_ok=True)
print("Hazir. Sonraki hucreye gec.")

In [ ]:
#@title 3. Repo Klonlama { display-mode: "form" }
import subprocess, os

REPO_URL = "https://github.com/Arda-Avci/AI-Publisher.git"
WORK_DIR = "/content/AI-Publisher"

print("GitHub PAT aliniyor...")
GITHUB_PAT = ""
try:
    from google.colab import userdata
    for key in ["GITHUB_PAT", "GITHUB_TOKEN", "GH_PAT", "GH_TOKEN"]:
        try:
            val = userdata.get(key)
            if val and len(val) > 10:
                GITHUB_PAT = val
                print(f"  Secrets: {key} (len={len(val)})")
                break
        except:
            pass
except:
    pass

if not GITHUB_PAT:
    print()
    print("=" * 60)
    print("GITHUB_PAT gerekli!")
    print("Colab Secrets ekle:")
    print("  1. Sol panel > Anahtar simgesi > Add new secret")
    print("  2. Name: GITHUB_PAT, Value: <token>, Notebook access: ON")
    print("  3. Runtime > Restart runtime")
    print("  4. Bu hucreyi yeniden calistir")
    print("=" * 60)
    raise ValueError("GITHUB_PAT gerekli")

print("\nRepo klonlaniyor...")
if os.path.exists(WORK_DIR):
    os.chdir(WORK_DIR)
    subprocess.run(["git", "fetch", "origin"], check=True, timeout=30)
    subprocess.run(["git", "reset", "--hard", "origin/main"], check=True, timeout=30)
    print("  Repo guncellendi.")
else:
    auth_url = REPO_URL.replace("https://", f"https://{GITHUB_PAT}@")
    r = subprocess.run(["git", "clone", auth_url, WORK_DIR], capture_output=True, text=True, timeout=60)
    if r.returncode != 0:
        print(f"HATA: {r.stderr[:300]}")
        print("Token gecersiz veya repo erisilemez.")
        raise RuntimeError("Git clone basarisiz")
    os.chdir(WORK_DIR)
    print("  Repo klonlandi.")

os.chdir(WORK_DIR + "/colab_docker")
print(f"  build_all.sh: {os.path.exists('build_all.sh')}")
print(f"  force_rebuild.sh: {os.path.exists('force_rebuild.sh')}")
subprocess.run(["chmod", "+x", "build_all.sh", "force_rebuild.sh"])
commit = subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(f"  Commit: {commit}")
print("Hazir. Sonraki hucreye gec.")

In [ ]:
#@title 4. Force-Rebuild: Base + Video Modelleri { display-mode: "form" }
import subprocess, os, time

os.chdir("/content/AI-Publisher/colab_docker")
VIDEO_MODELS = ["base", "cogvideox", "wan", "ltx", "hunyuan", "svd", "animatediff", "wan25"]

print("Force-rebuild basliyor...")
print(f"Modeller: {' '.join(VIDEO_MODELS)}")
print("=" * 40)

t0 = time.time()
r = subprocess.run(
    ["bash", "force_rebuild.sh"] + VIDEO_MODELS,
    capture_output=True, text=True, timeout=72000
)
t = (time.time() - t0) / 60

print(f"Cikis: {r.returncode}, Sure: {t:.1f} dk")
print("=" * 40)
if r.returncode != 0:
    print("\nHATA - STDERR:")
    for line in (r.stderr or "").split("\n"):
        if line.strip():
            print(f"  {line}")
    print("\nSTDOUT (son 50 satir):")
    for line in (r.stdout or "").split("\n")[-50:]:
        if line.strip():
            print(f"  {line}")
else:
    print("Build basarili!")

In [ ]:
#@title 5. Incremental Build: Ses/Efekt Modelleri { display-mode: "form" }
import subprocess, os, time

os.chdir("/content/AI-Publisher/colab_docker")
print("Incremental build basliyor (Drive'da var olan atlanir)")
print("=" * 40)

t0 = time.time()
r = subprocess.run(["bash", "build_all.sh"], capture_output=True, text=True, timeout=43200)
t = (time.time() - t0) / 60

print(f"Cikis: {r.returncode}, Sure: {t:.1f} dk")
if r.returncode != 0:
    print("\nSTDERR:")
    for line in (r.stderr or "").split("\n")[-20:]:
        if line.strip():
            print(f"  {line}")
else:
    print("Build basarili!")

In [ ]:
#@title 6. Imaj Dogrulama { display-mode: "form" }
import subprocess, os, datetime, glob

DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks/docker/images"
MIN_SIZE = 100 * 1024 * 1024

if not os.path.exists(DRIVE_DIR):
    print(f"Drive dizini bulunamadi: {DRIVE_DIR}")
else:
    files = sorted(glob.glob(os.path.join(DRIVE_DIR, "*.tar.gz")))
    print(f"Toplam {len(files)} imaj dosyasi\n")
    
    all_models = ["base", "cogvideox", "wan", "ltx", "hunyuan", "svd",
                  "animatediff", "wan25", "xtts", "audioldm2", "wav2lip",
                  "musetalk", "whisper", "stablediffusion", "kokorotts",
                  "f5tts", "lora-trainer"]
    
    drive_map = {os.path.basename(f).replace(".tar.gz", ""): f for f in files}
    for model in all_models:
        if model in drive_map:
            sz = os.path.getsize(drive_map[model])
            sz_mb = sz / (1024 * 1024)
            mtime = datetime.datetime.fromtimestamp(os.path.getmtime(drive_map[model]))
            status = "OK" if sz >= MIN_SIZE else "KUCUK"
            print(f"  [{status:6s}] {model:20s} {sz_mb:8.1f} MB  {mtime:%Y-%m-%d %H:%M}")
        else:
            print(f"  [EKSIK  ] {model:20s}")
    
    missing = [m for m in all_models if m not in drive_map]
    small = [m for m in all_models if m in drive_map and os.path.getsize(drive_map[m]) < MIN_SIZE]
    print()
    if not missing and not small:
        print("TUM IMAJLAR HAZIR")
    else:
        if missing: print(f"EKSIK: {', '.join(missing)}")
        if small: print(f"KUCUK: {', '.join(small)}")

## Build Tamamlandi

Sonraki adimlar:
1. Imajlari yerel makineye indir: Drive > `Colab Notebooks/docker/images/` > kopyala
2. `docker compose up -d` ile baslat
3. `curl http://localhost:5001/health` ile dogrula

### Port Haritasi
- 5001 cogvideox (Video I2V)
- 5002 xtts (TTS)
- 5003 audioldm2 (SFX)
- 5004 wav2lip (Lip-Sync)
- 5005 musetalk (Lip-Sync alt)
- 5006 whisper (STT)
- 5007 stablediffusion (Gorsel)
- 5008 wan (Video T2V)
- 5009 ltx (Video I2V)
- 5010 hunyuan (Video)
- 5011 kokorotts (TTS)
- 5012 svd (Video I2V)
- 5013 animatediff (Video)
- 5014 wan25 (Video)
- 5015 f5tts (TTS)
- 5016 lora-trainer (LoRA)